In [3]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT 
from psycopg2.extras import execute_batch

import pandas as pd

In [4]:
# with open('pwd.txt', 'w') as file:
#     file.write(input('Enter the password: '))
with open('pwd.txt', 'r') as file:
    pwd = file.read()
conn = psycopg2.connect(
        dbname='postgres',
        user='postgres',
        password=pwd,
        host='localhost',
        port='5432')

In [9]:
cur.execute("CREATE DATABASE two;")

In [23]:
cur.execute("DROP TABLE flights_schedule;")

In [8]:
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

In [7]:
conn.rollback()

In [5]:
cur = conn.cursor()

In [24]:
df = pd.read_csv('flight_data_.csv')

mapping = {'int64' : 'INTEGER',
        'str' : 'VARCHAR(100)',
        'object': 'VARCHAR(100)',
        'float64' : 'NUMERIC(10, 2)',
        'bool' : 'BOOLEAN'}
postgre_dtypes = df.dtypes.reset_index(drop=False).iloc[:, 1].astype(str)
postgre_dtypes = postgre_dtypes.apply(lambda x: mapping[x])

cols = ', '.join([f'"{col}"' for col in df.columns])
columns = ''
for c, t in zip(df.columns, postgre_dtypes):
    columns += f'"{c}" {t}, '
columns = columns.strip(', ')
vals = ', '.join(['%s'] * len(df.columns))

cur.execute(f'CREATE TABLE data ({columns});')
cur.executemany(f'INSERT INTO data ({cols}) VALUES ({vals});', [row for row in df.itertuples(index=False)])

# дату рейса переводим в тип даты
cur.execute('''ALTER TABLE data 
ALTER COLUMN "Departure Date" TYPE TIMESTAMP 
USING TO_TIMESTAMP("Departure Date", 'YYYY-MM-DD HH24:MI:SS');''')

In [32]:
import pandas as pd

# Загружаем всю таблицу data_clean (или с лимитом)
df_data_clean = pd.read_sql_query("SELECT * FROM data_clean LIMIT 20;", conn)

# Показываем информацию о таблице
print("=" * 80)
print("📊 ТАБЛИЦА: data_clean")
print("=" * 80)

# 1. Структура таблицы (типы данных)
print("\n--- СТРУКТУРА ТАБЛИЦЫ ---")
print(df_data_clean.dtypes.to_string())
print()

# 2. Первые 20 строк
print("--- ПЕРВЫЕ 20 СТРОК ---")
print(df_data_clean.to_string())
print()

# 3. Общая статистика
print("--- ОБЩАЯ СТАТИСТИКА ---")
print(f"Всего строк: {pd.read_sql_query('SELECT COUNT(*) FROM data_clean;', conn).iloc[0, 0]}")
print(f"Всего колонок: {len(df_data_clean.columns)}")
print()

# 4. Проверка колонки "Delay Minutes Clean" (создана ли)
print("--- ПРОВЕРКА КОЛОНКИ 'Delay Minutes Clean' ---")
df_delay = pd.read_sql_query("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT("Delay Minutes Clean") AS non_null_count,
        COUNT(CASE WHEN "Delay Minutes Clean" = 0 THEN 1 END) AS zero_count,
        MIN("Delay Minutes Clean") AS min_delay,
        MAX("Delay Minutes Clean") AS max_delay,
        ROUND(AVG("Delay Minutes Clean"), 2) AS avg_delay
    FROM data_clean;
""", conn)
print(df_delay.to_string())
print()

# 5. Сравнение исходной и новой колонок задержки
print("--- СРАВНЕНИЕ 'Delay Minutes' и 'Delay Minutes Clean' ---")
df_compare = pd.read_sql_query("""
    SELECT 
        "Delay Minutes",
        "Delay Minutes Clean",
        CASE 
            WHEN "Delay Minutes" IS NULL THEN 'NULL заменен на 0'
            ELSE 'Оригинальное значение'
        END AS status
    FROM data_clean
    WHERE "Delay Minutes" IS NULL OR "Delay Minutes" = 0
    LIMIT 10;
""", conn)
print(df_compare.to_string())
print()

# 6. Статистика по типам данных
print("--- ТИПЫ ДАННЫХ ВСЕХ КОЛОНОК ---")
for col in df_data_clean.columns:
    dtype = df_data_clean[col].dtype
    non_null = df_data_clean[col].count()
    print(f"{col:25} | {str(dtype):15} | Не NULL: {non_null}")

📊 ТАБЛИЦА: data_clean

--- СТРУКТУРА ТАБЛИЦЫ ---
Departure City                      str
Arrival City                        str
Departure Date           datetime64[us]
Flight Duration                 float64
Delay Minutes                     int64
Customer ID                       int64
Name                                str
Booking Class                       str
Frequent Flyer Status               str
Route                               str
Ticket Price                    float64
Competitor Price                float64
Demand                          float64
Origin                              str
Destination                         str
Profitability                   float64
Loyalty Points                    int64
Churned                            bool
Delay Minutes Clean               int64

--- ПЕРВЫЕ 20 СТРОК ---
         Departure City      Arrival City      Departure Date  Flight Duration  Delay Minutes  Customer ID                Name Booking Class Frequent Flyer Status    

C:\Users\I\AppData\Local\Temp\ipykernel_912\1601425900.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_data_clean = pd.read_sql_query("SELECT * FROM data_clean LIMIT 20;", conn)
C:\Users\I\AppData\Local\Temp\ipykernel_912\1601425900.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(f"Всего строк: {pd.read_sql_query('SELECT COUNT(*) FROM data_clean;', conn).iloc[0, 0]}")
C:\Users\I\AppData\Local\Temp\ipykernel_912\1601425900.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_delay = pd.read_sql_query("""

In [25]:
# создаем "очищенную таблицу", на случай если нет значения в столбце задержки (для использования COALESCE)
cur.execute("""
CREATE TABLE data_clean AS
SELECT *, COALESCE("Delay Minutes", 0) AS "Delay Minutes Clean" FROM data;""")

# создаем таблицу расписания рейсов
cur.execute('''
CREATE TABLE flights_schedule AS
SELECT
    ROW_NUMBER() OVER (ORDER BY "Route", "Departure Date") AS "Flight ID",
    "Route",
    "Departure Date",
    MAX("Delay Minutes Clean") AS "Delay Minutes Clean",
    COUNT(*) AS "Passenger Count",
    AVG("Ticket Price") AS "Avg Ticket Price"
FROM data_clean
GROUP BY "Route", "Departure Date";
''')

# создаем таблицу пассажиры
cur.execute('''
CREATE TABLE customers AS
SELECT DISTINCT ON ("Customer ID")
    "Customer ID",
    "Name",
    COUNT(*) OVER(PARTITION BY "Customer ID") AS "Flights count",
    "Frequent Flyer Status",
    "Loyalty Points"
FROM data_clean
ORDER BY "Customer ID", "Departure Date" DESC;
''')

In [27]:
import pandas as pd

# 1. Выводим data_clean (первые 10 строк)
print("=" * 80)
print("ТАБЛИЦА: data_clean")
print("=" * 80)
df_data_clean = pd.read_sql_query("SELECT * FROM data_clean LIMIT 10;", conn)
print(df_data_clean.to_string())
print(f"\nВсего строк в data_clean: {pd.read_sql_query('SELECT COUNT(*) FROM data_clean;', conn).iloc[0, 0]}")
print()

# 2. Выводим flights_schedule (первые 15 строк)
print("=" * 80)
print("ТАБЛИЦА: flights_schedule")
print("=" * 80)
df_flights = pd.read_sql_query("""
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date"::DATE AS date,
        "Passenger Count",
        "Delay Minutes Clean",
        "Avg Ticket Price"
    FROM flights_schedule 
    ORDER BY "Departure Date"
    LIMIT 15;
""", conn)
print(df_flights.to_string())
print(f"\nВсего рейсов: {pd.read_sql_query('SELECT COUNT(*) FROM flights_schedule;', conn).iloc[0, 0]}")
print()

# 3. Выводим customers (первые 10 строк)
print("=" * 80)
print("ТАБЛИЦА: customers")
print("=" * 80)
df_customers = pd.read_sql_query("""
    SELECT 
        "Customer ID",
        "Name",
        "Flights count",
        "Frequent Flyer Status",
        "Loyalty Points"
    FROM customers 
    ORDER BY "Flights count" DESC
    LIMIT 10;
""", conn)
print(df_customers.to_string())

# Исправлено: сумма всех перелетов (а не количество пассажиров)
total_flights = pd.read_sql_query('SELECT SUM("Flights count") FROM customers;', conn).iloc[0, 0]
print(f"\n📊 Всего перелетов (сумма Flights count): {total_flights}")
print()

# 4. Статистика по статусам (дополнительно)
print("=" * 80)
print("СТАТИСТИКА ПО СТАТУСАМ (customers)")
print("=" * 80)
df_status = pd.read_sql_query("""
    SELECT 
        "Frequent Flyer Status",
        COUNT(*) AS customers,
        ROUND(AVG("Flights count"), 2) AS avg_flights,
        ROUND(AVG("Loyalty Points"), 2) AS avg_points
    FROM customers
    GROUP BY "Frequent Flyer Status"
    ORDER BY customers DESC;
""", conn)
print(df_status.to_string())
print()

# 5. Общая статистика по всем таблицам
print("=" * 80)
print("ОБЩАЯ СТАТИСТИКА")
print("=" * 80)
df_stats = pd.read_sql_query("""
    SELECT 
        'data_clean' AS table_name, 
        COUNT(*) AS rows,
        COUNT(DISTINCT "Customer ID") AS unique_customers,
        COUNT(DISTINCT "Route") AS unique_routes
    FROM data_clean
    UNION ALL
    SELECT 
        'flights_schedule', 
        COUNT(*),
        NULL,
        COUNT(DISTINCT "Route")
    FROM flights_schedule
    UNION ALL
    SELECT 
        'customers', 
        COUNT(*),
        COUNT(*),
        NULL
    FROM customers;
""", conn)
print(df_stats.to_string())
print()

# 6. Дополнительная проверка: сравнение сумм
print("=" * 80)
print("ПРОВЕРКА СОВПАДЕНИЯ")
print("=" * 80)
df_check = pd.read_sql_query("""
    SELECT 
        (SELECT SUM("Flights count") FROM customers) AS total_flights_customers,
        (SELECT COUNT(*) FROM data_clean) AS total_rows_data_clean,
        (SELECT SUM("Passenger Count") FROM flights_schedule) AS total_passengers_flights,
        CASE 
            WHEN (SELECT SUM("Flights count") FROM customers) = (SELECT COUNT(*) FROM data_clean) 
            THEN '✅ СОВПАДАЕТ' 
            ELSE '❌ НЕ СОВПАДАЕТ' 
        END AS status
""", conn)
print(df_check.to_string())

ТАБЛИЦА: data_clean
       Departure City      Arrival City      Departure Date  Flight Duration  Delay Minutes  Customer ID               Name Booking Class Frequent Flyer Status    Route  Ticket Price  Competitor Price  Demand Origin Destination  Profitability  Loyalty Points  Churned  Delay Minutes Clean
0          Wilsonstad    Lake Johnmouth 2023-05-02 20:11:09             1.28            120         3769      Daniel Oliver      Business                  Gold  MEL-BNE        370.64            382.95   -0.93    MEL         LHR           0.63            4245     True                  120
1           New Brent        Port Wanda 2023-04-21 00:10:14             1.28             35         3529       Deborah Hall       Economy              Platinum  BNE-SYD        114.53            394.58   -1.01    MEL         SIN           1.27             833     True                   35
2  South Samanthaberg    Lake Meganside 2023-05-12 15:16:31             0.72             67         1303         

C:\Users\I\AppData\Local\Temp\ipykernel_912\2494294482.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_data_clean = pd.read_sql_query("SELECT * FROM data_clean LIMIT 10;", conn)
C:\Users\I\AppData\Local\Temp\ipykernel_912\2494294482.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(f"\nВсего строк в data_clean: {pd.read_sql_query('SELECT COUNT(*) FROM data_clean;', conn).iloc[0, 0]}")
C:\Users\I\AppData\Local\Temp\ipykernel_912\2494294482.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_flights = pd.re

In [ ]:
# делаем таблицу билетов (чтобы потом воспользоваться разными джоинами)
# эти джоины нам ничего не дадут, так как все сделано из одного общего датасета, в котором все строки заполнены
cur.execute('''CREATE TABLE tickets AS
SELECT 
    f."Flight ID",           
    d."Customer ID",         
    d."Booking Class",       
    d."Ticket Price",       
    d."Competitor Price",    
    d."Demand",              
    d."Profitability"     
FROM data_clean d
JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')
# здесь мы типо создали билеты, что каждый пассажир получил номер рейса(у кого нет рейса из пассажиров или если рейс пустой, то пропущено
# такие вот типо билеты, здесь только те у кого есть билеты

In [ ]:
# ипсользуем левый джоин, в таком случае не все пассажиры получили бы номер рейса, если бы, к примеру, в расписании рейса отсутствовали рейсы некоторые
# таким образом у нас есть все клиенты-пассажиры + номера существующих рейсов
cur.execute('''CREATE TABLE tickets_left AS
SELECT 
    f."Flight ID",
    d."Customer ID",
    d."Booking Class",
    d."Ticket Price"
FROM data_clean d
LEFT JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')

In [ ]:
# здесь парвый джоин оставляет всех пассажиров, которые имеют рейсы из расписания
cur.execute('''CREATE TABLE flights_with_passengers AS
SELECT 
    f."Flight ID",
    f."Route",
    f."Departure Date",
    f."Passenger Count",
    d."Customer ID"
FROM data_clean d
RIGHT JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')

In [ ]:
# здесь берем все записи, там где нет пассажиров у рейса + нет рейса у пассажира + есть и рейс, и пассажир(билет полный)
cur.execute('''CREATE TABLE full_join AS
SELECT 
    f."Flight ID",
    d."Customer ID",
    d."Booking Class",
    COALESCE(f."Route", d."Route") AS "Route"
FROM data_clean d
FULL OUTER JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')

In [28]:
# таблица со скользящими средними и стандартным отклонением за 3 дня
cur.execute('''CREATE TABLE rolling_3days AS
WITH rolling_calc AS (
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date",
        "Passenger Count",
        COUNT(*) OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS window_size,
        ROUND(AVG("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling Avg 3 Days",
        ROUND(STDDEV("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling StdDev 3 Days"
    FROM flights_schedule
)
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Passenger Count",
    CASE WHEN window_size = 3 THEN "Rolling Avg 3 Days" ELSE NULL END AS "Rolling Avg 3 Days",
    CASE WHEN window_size = 3 THEN "Rolling StdDev 3 Days" ELSE NULL END AS "Rolling StdDev 3 Days",
    CASE 
        WHEN window_size = 3 AND "Rolling Avg 3 Days" > LAG("Rolling Avg 3 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'INCREASING'
        WHEN window_size = 3 AND "Rolling Avg 3 Days" < LAG("Rolling Avg 3 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'DECLINING'
        WHEN window_size = 3 THEN 'STABLE'
        ELSE NULL
    END AS "Demand Trend"
FROM rolling_calc
WHERE window_size = 3;''')

In [29]:
# таблица со скользящими средними и стандартным отклонением за 6 дней
cur.execute('''CREATE TABLE rolling_6days AS
WITH rolling_calc AS (
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date",
        "Passenger Count",
        COUNT(*) OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS window_size,
        ROUND(AVG("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling Avg 6 Days",
        ROUND(STDDEV("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling StdDev 6 Days"
    FROM flights_schedule
)
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Passenger Count",
    CASE WHEN window_size = 6 THEN "Rolling Avg 6 Days" ELSE NULL END AS "Rolling Avg 6 Days",
    CASE WHEN window_size = 6 THEN "Rolling StdDev 6 Days" ELSE NULL END AS "Rolling StdDev 6 Days",
    CASE 
        WHEN window_size = 6 AND "Rolling Avg 6 Days" > LAG("Rolling Avg 6 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'INCREASING'
        WHEN window_size = 6 AND "Rolling Avg 6 Days" < LAG("Rolling Avg 6 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'DECLINING'
        WHEN window_size = 6 THEN 'STABLE'
        ELSE NULL
    END AS "Demand Trend"
FROM rolling_calc
WHERE window_size = 6;''')

In [30]:
# таблица со скользящими средними и стандартным отклонением за 9 дней
cur.execute('''CREATE TABLE rolling_9days AS
WITH rolling_calc AS (
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date",
        "Passenger Count",
        COUNT(*) OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ) AS window_size,
        ROUND(AVG("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling Avg 9 Days",
        ROUND(STDDEV("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling StdDev 9 Days"
    FROM flights_schedule
)
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Passenger Count",
    CASE WHEN window_size = 9 THEN "Rolling Avg 9 Days" ELSE NULL END AS "Rolling Avg 9 Days",
    CASE WHEN window_size = 9 THEN "Rolling StdDev 9 Days" ELSE NULL END AS "Rolling StdDev 9 Days",
    CASE 
        WHEN window_size = 9 AND "Rolling Avg 9 Days" > LAG("Rolling Avg 9 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'INCREASING'
        WHEN window_size = 9 AND "Rolling Avg 9 Days" < LAG("Rolling Avg 9 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'DECLINING'
        WHEN window_size = 9 THEN 'STABLE'
        ELSE NULL
    END AS "Demand Trend"
FROM rolling_calc
WHERE window_size = 9;''')

In [31]:
import pandas as pd

# Загружаем таблицу в DataFrame
df_rolling = pd.read_sql_query("SELECT * FROM rolling_3days LIMIT 20;", conn)

# Показываем
print(df_rolling.to_string())
# или
display(df_rolling)  # если в Jupyter

    Flight ID    Route      Departure Date  Passenger Count  Rolling Avg 3 Days  Rolling StdDev 3 Days Demand Trend
0           3  BNE-SYD 2023-04-25 21:17:18               12               12.67                   1.15       STABLE
1           4  BNE-SYD 2023-04-30 11:39:08               10               11.33                   1.15    DECLINING
2           5  BNE-SYD 2023-05-01 01:09:21               14               12.00                   2.00   INCREASING
3           6  BNE-SYD 2023-05-05 15:27:08               17               13.67                   3.51   INCREASING
4           7  BNE-SYD 2023-05-06 12:36:14               14               15.00                   1.73   INCREASING
5           8  BNE-SYD 2023-05-07 23:32:44               16               15.67                   1.53   INCREASING
6           9  BNE-SYD 2023-05-10 10:06:19               15               15.00                   1.00    DECLINING
7          10  BNE-SYD 2023-05-11 22:47:43               15             

C:\Users\I\AppData\Local\Temp\ipykernel_912\3561125721.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_rolling = pd.read_sql_query("SELECT * FROM rolling_3days LIMIT 20;", conn)


,Flight ID,Route,Departure Date,Passenger Count,Rolling Avg 3 Days,Rolling StdDev 3 Days,Demand Trend
0,3,BNE-SYD,2023-04-25 21:17:18,12,12.67,1.15,STABLE
1,4,BNE-SYD,2023-04-30 11:39:08,10,11.33,1.15,DECLINING
2,5,BNE-SYD,2023-05-01 01:09:21,14,12.00,2.00,INCREASING
3,6,BNE-SYD,2023-05-05 15:27:08,17,13.67,3.51,INCREASING
4,7,BNE-SYD,2023-05-06 12:36:14,14,15.00,1.73,INCREASING
5,8,BNE-SYD,2023-05-07 23:32:44,16,15.67,1.53,INCREASING
6,9,BNE-SYD,2023-05-10 10:06:19,15,15.00,1.00,DECLINING
7,10,BNE-SYD,2023-05-11 22:47:43,15,15.33,0.58,INCREASING
8,11,BNE-SYD,2023-05-13 10:40:25,15,15.00,0.00,DECLINING
9,12,BNE-SYD,2023-05-15 04:40:39,14,14.67,0.58,DECLINING


In [27]:
import pandas as pd

# Основная статистика по рейсам
df_flights = pd.read_sql_query("""
SELECT 
    COUNT(*) AS total_flights,
    COUNT(DISTINCT "Route") AS unique_routes,
    MIN("Departure Date") AS first_flight,
    MAX("Departure Date") AS last_flight,
    ROUND(AVG("Passenger Count"), 2) AS avg_passengers,
    MIN("Passenger Count") AS min_passengers,
    MAX("Passenger Count") AS max_passengers
FROM flights_schedule;
""", conn)
print("=== Статистика по рейсам ===")
print(df_flights)
print("\n")

# Посмотрим 10 рейсов
df_flights_sample = pd.read_sql_query("""
SELECT 
    "Flight ID",
    "Route",
    "Departure Date"::DATE AS date,
    "Passenger Count",
    "Delay Minutes Clean",
    "Avg Ticket Price"
FROM flights_schedule 
ORDER BY "Departure Date" 
LIMIT 10;
""", conn)
print("=== Первые 10 рейсов ===")
print(df_flights_sample.to_string())

=== Статистика по рейсам ===
   total_flights  unique_routes        first_flight         last_flight  \
0            100              3 2023-04-18 01:13:37 2023-06-16 21:25:28   

   avg_passengers  min_passengers  max_passengers  
0            14.0               7              22  


=== Первые 10 рейсов ===
   Flight ID    Route        date  Passenger Count  Delay Minutes Clean  Avg Ticket Price
0         68  SYD-MEL  2023-04-18               20                   11            440.65
1         69  SYD-MEL  2023-04-18               18                   77            218.75
2         33  MEL-BNE  2023-04-18               20                   71            258.60
3         34  MEL-BNE  2023-04-19               10                   58            170.78
4          1  BNE-SYD  2023-04-21               14                   35            114.53
5         70  SYD-MEL  2023-04-21               11                  108            304.87
6         35  MEL-BNE  2023-04-22               20         

C:\Users\I\AppData\Local\Temp\ipykernel_6920\366846840.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_flights = pd.read_sql_query("""
C:\Users\I\AppData\Local\Temp\ipykernel_6920\366846840.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_flights_sample = pd.read_sql_query("""
